# 08 — Refatoração Multidimensional do Score endurecimento (análise nova)

O score composto (`purificacao_composto`) é a média simples dos 10 indicadores.
A análise PCA (notebook 07) mostrou que isso captura apenas 53.7% da variância.
Este notebook propõe sub-scores baseados na estrutura dos dados.

**Fonte:** `data/processed/corpus_dataset.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import kruskal, spearmanr

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/processed/corpus_dataset.csv')

INDICATORS = ['desincorporacao', 'rigidez_postural', 'dessexualizacao',
              'uniformizacao_facial', 'heraldizacao', 'enquadramento_arquitetonico',
              'apagamento_narrativo', 'monocromatizacao', 'serialidade', 'inscricao_estatal']

INDICATOR_LABELS = {
    'desincorporacao': 'Desincorporação', 'rigidez_postural': 'Rigidez postural',
    'dessexualizacao': 'Dessexualização', 'uniformizacao_facial': 'Uniformização facial',
    'heraldizacao': 'Heraldicização', 'enquadramento_arquitetonico': 'Enquadramento arq.',
    'apagamento_narrativo': 'Apagamento narrativo', 'monocromatizacao': 'Monocromatização',
    'serialidade': 'Serialidade', 'inscricao_estatal': 'Inscrição estatal',
}

df_c = df.dropna(subset=INDICATORS).copy()
X = df_c[INDICATORS].values

print(f"Itens com indicadores completos: {len(df_c)}")
print(f"Composite atual (média simples): min={df_c['purificacao_composto'].min():.2f}, max={df_c['purificacao_composto'].max():.2f}, mean={df_c['purificacao_composto'].mean():.2f}")

## 8.1 Definição dos Sub-Scores

Baseado na estrutura PCA (notebook 07), propomos 3 sub-scores:

| Sub-Score | Indicadores | Justificativa PCA |
|-----------|-------------|-------------------|
| **endurecimento_CORE** | 8 indicadores (exclui monocromatização + serialidade) | PC1 — dimensão geral de endurecimento |
| **MONOCROMATIZAÇÃO** | monocromatização (isolada) | PC3 — loading 0.87, dimensão independente |
| **FORMALIZAÇÃO_BUR.** | serialidade + inscrição estatal | PC2 — eixo burocrático |

O composite original (média dos 10) é mantido para compatibilidade.

In [ ]:
# Define sub-score groups
CORE_8 = ['desincorporacao', 'rigidez_postural', 'dessexualizacao',
          'uniformizacao_facial', 'heraldizacao', 'enquadramento_arquitetonico',
          'apagamento_narrativo', 'inscricao_estatal']
MONO_1 = ['monocromatizacao']
BUR_2 = ['serialidade', 'inscricao_estatal']

# Compute sub-scores
df_c['endurecimento_core'] = df_c[CORE_8].mean(axis=1)
df_c['monocromatizacao_score'] = df_c[MONO_1].mean(axis=1)
df_c['formalizacao_bur'] = df_c[BUR_2].mean(axis=1)

# Also compute PCA factor scores for reference
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
df_c['PC1'] = X_pca[:, 0]
df_c['PC2'] = X_pca[:, 1]
df_c['PC3'] = X_pca[:, 2]

print("Sub-scores computed:")
print(f"  endurecimento_CORE (8 ind): mean={df_c['endurecimento_core'].mean():.2f}, std={df_c['endurecimento_core'].std():.2f}")
print(f"  MONOCROMATIZAÇÃO (1 ind):    mean={df_c['monocromatizacao_score'].mean():.2f}, std={df_c['monocromatizacao_score'].std():.2f}")
print(f"  FORMALIZAÇÃO_BUR (2 ind):    mean={df_c['formalizacao_bur'].mean():.2f}, std={df_c['formalizacao_bur'].std():.2f}")
print(f"  Composite original (10 ind): mean={df_c['purificacao_composto'].mean():.2f}, std={df_c['purificacao_composto'].std():.2f}")

# Correlation between sub-scores
sub_cols = ['endurecimento_core', 'monocromatizacao_score', 'formalizacao_bur', 'purificacao_composto']
print(f"\nCorrelação entre sub-scores (Spearman):")
print(df_c[sub_cols].corr(method='spearman').round(3).to_string())

## 8.2 Distribuição dos Sub-Scores por Regime

In [ ]:
regime_order = ['fundacional', 'normativo', 'militar', 'contra-alegoria']
regime_colors = {'fundacional': '#4C72B0', 'normativo': '#DD8452',
                 'militar': '#C44E52', 'contra-alegoria': '#55A868'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title in zip(axes,
    ['endurecimento_core', 'monocromatizacao_score', 'formalizacao_bur'],
    ['endurecimento Core', 'Monocromatização', 'Formalização Burocrática']):
    
    data = [df_c[df_c['regime_iconocratico'] == r][col].values for r in regime_order if r in df_c['regime_iconocratico'].values]
    labels = [r for r in regime_order if r in df_c['regime_iconocratico'].values]
    colors = [regime_colors[r] for r in labels]
    
    parts = ax.violinplot(data, showmeans=True, showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.6)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=30)
    ax.set_ylabel('Score (0–3)')
    ax.set_title(title)

plt.suptitle('Sub-Scores por Regime Iconocrático', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/fig_16_subscore_by_regime.png', dpi=150, bbox_inches='tight')
plt.show()

# Print stats
print("Média por regime:")
for col in ['endurecimento_core', 'monocromatizacao_score', 'formalizacao_bur', 'purificacao_composto']:
    print(f"\n  {col}:")
    for r in regime_order:
        subset = df_c[df_c['regime_iconocratico'] == r][col]
        if len(subset) > 0:
            print(f"    {r:20s}: {subset.mean():.2f} ± {subset.std():.2f} (n={len(subset)})")

## 8.3 Kruskal-Wallis: Sub-Scores × Regimes (reteste)

In [ ]:
regimes_present = [r for r in regime_order if r in df_c['regime_iconocratico'].values]

print("Kruskal-Wallis por sub-score (regimes):")
for col in ['endurecimento_core', 'monocromatizacao_score', 'formalizacao_bur', 'purificacao_composto']:
    groups = [df_c[df_c['regime_iconocratico'] == r][col].values for r in regimes_present]
    groups = [g for g in groups if len(g) >= 2]
    if len(groups) >= 2:
        stat, p = kruskal(*groups)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f"  {col:30s}: H={stat:.2f}, p={p:.2e} {sig}")
    else:
        print(f"  {col:30s}: insuficiente (menos de 2 grupos com n≥2)")

## 8.4 Scatter: Core vs Monocromatização (colored by regime)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Core vs Monocromatização
ax = axes[0]
for regime, color in regime_colors.items():
    mask = df_c['regime_iconocratico'] == regime
    ax.scatter(df_c.loc[mask, 'endurecimento_core'], df_c.loc[mask, 'monocromatizacao_score'],
               c=color, label=regime, alpha=0.7, s=50, edgecolors='white', linewidth=0.5)
ax.set_xlabel('endurecimento Core (8 ind)')
ax.set_ylabel('Monocromatização')
ax.set_title('Core × Monocromatização por Regime')
ax.legend(title='Regime')

# Core vs Formalização Burocrática
ax = axes[1]
for regime, color in regime_colors.items():
    mask = df_c['regime_iconocratico'] == regime
    ax.scatter(df_c.loc[mask, 'endurecimento_core'], df_c.loc[mask, 'formalizacao_bur'],
               c=color, label=regime, alpha=0.7, s=50, edgecolors='white', linewidth=0.5)
ax.set_xlabel('endurecimento Core (8 ind)')
ax.set_ylabel('Formalização Burocrática (serialidade + inscrição)')
ax.set_title('Core × Formalização Burocrática por Regime')
ax.legend(title='Regime')

plt.tight_layout()
plt.savefig('../data/processed/fig_17_subscore_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 8.5 Quem muda de posição? Itens com maior divergência entre composite e sub-scores

In [ ]:
# Items where monocromatização most inflates or deflates the composite
df_c['mono_contribution'] = df_c['monocromatizacao_score'] - df_c['endurecimento_core']
df_c['composite_vs_core'] = df_c['purificacao_composto'] - df_c['endurecimento_core']

print("Top 5 itens MAIS inflados pela monocromatização (composite >> core):")
top_inflated = df_c.nlargest(5, 'composite_vs_core')[['id', 'title', 'regime_iconocratico', 'endurecimento_core', 'monocromatizacao_score', 'purificacao_composto', 'composite_vs_core']]
for _, row in top_inflated.iterrows():
    print(f"  {row['id']:20s} core={row['endurecimento_core']:.2f}  mono={row['monocromatizacao_score']:.1f}  comp={row['purificacao_composto']:.2f}  Δ={row['composite_vs_core']:+.2f}  [{row['regime_iconocratico']}]  {row['title'][:50]}")

print("\nTop 5 itens MAIS deflacionados (composite << core):")
top_deflated = df_c.nsmallest(5, 'composite_vs_core')[['id', 'title', 'regime_iconocratico', 'endurecimento_core', 'monocromatizacao_score', 'purificacao_composto', 'composite_vs_core']]
for _, row in top_deflated.iterrows():
    print(f"  {row['id']:20s} core={row['endurecimento_core']:.2f}  mono={row['monocromatizacao_score']:.1f}  comp={row['purificacao_composto']:.2f}  Δ={row['composite_vs_core']:+.2f}  [{row['regime_iconocratico']}]  {row['title'][:50]}")

# Histogram of the gap
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_c['composite_vs_core'], bins=20, edgecolor='white', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Composite − Core (diferença)')
ax.set_ylabel('Contagem')
ax.set_title('Distribuição: Composite (10 ind) − Core (8 ind)')
plt.tight_layout()
plt.savefig('../data/processed/fig_18_composite_vs_core.png', dpi=150, bbox_inches='tight')
plt.show()

## 8.6 Export: CSV com sub-scores para backfill

In [ ]:
# Export sub-scores for integration into the pipeline
export_cols = ['id', 'endurecimento_core', 'monocromatizacao_score', 'formalizacao_bur', 'PC1', 'PC2', 'PC3']
export = df_c[export_cols].copy()
for col in ['endurecimento_core', 'monocromatizacao_score', 'formalizacao_bur']:
    export[col] = export[col].round(3)
for col in ['PC1', 'PC2', 'PC3']:
    export[col] = export[col].round(4)

export.to_csv('../data/processed/subscores.csv', index=False)
print(f"Exported {len(export)} rows to data/processed/subscores.csv")
print(export.head(10).to_string())

## 8.7 Síntese e Recomendação

### O que este notebook mostra:
- Os 3 sub-scores capturam dimensões que o composite esconde
- A monocromatização infla/deflaciona o composite para itens específicos
- Kruskal-Wallis com sub-scores pode revelar diferenças de regime que o composite obscurece

### Próximos passos:
1. Atualizar `code_purification.py` para computar sub-scores no momento da codificação
2. Adicionar colunas ao `corpus_dataset.csv`: `endurecimento_core`, `monocromatizacao_score`, `formalizacao_bur`
3. Re-executar notebooks 02 (Kruskal-Wallis) e 03 (regressão) com sub-scores
4. Discutir com orientador: manter composite para compatibilidade, usar sub-scores nas análises?